# ragnhild — FLUX LoRA

Generated by `STUDIO/lora/_toolkit/run_train_colab.rb`. Edits belong in the
generator; this file is overwritten.

**Runtime → Change runtime type → T4 GPU** before running, or it trains on a
CPU and never finishes.

Set `HF_TOKEN` in the sidebar (🔑) to a Hugging Face token that has accepted
the [FLUX.1-dev licence](https://huggingface.co/black-forest-labs/FLUX.1-dev).

1000 steps, sampling twelve lighting setups every
500. A free session is capped near 12 h and disconnects
when idle, so this is likely more than one sitting — checkpoints go to Drive
and re-running resumes from the newest. Source: `anon987654321/pub4`.


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("ok: HF_TOKEN from Colab secrets")
except Exception:
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (read scope, FLUX.1-dev licence accepted): ")
assert os.environ.get("HF_TOKEN"), "no HF token"

In [ ]:
# Checkpoints and portraits go to Drive so a disconnect costs the session
# and not the training. Re-running the notebook resumes from the newest.
import os
from google.colab import drive
drive.mount("/content/drive")
os.environ["LORA_PERSIST_DIR"] = "/content/drive/MyDrive/lora/ragnhild"
os.makedirs(os.environ["LORA_PERSIST_DIR"], exist_ok=True)

In [ ]:
import subprocess
subprocess.run("apt-get -qq update && apt-get -qq install -y ruby git",
               shell=True, check=True)
if not os.path.isdir("/content/pub4/.git"):
    subprocess.run(["git", "clone", "--branch", "main", "--depth", "1",
                    "https://github.com/anon987654321/pub4.git", "/content/pub4"], check=True)
else:
    subprocess.run(["git", "-C", "/content/pub4", "pull", "--ff-only"], check=True)
# Runtime type is a menu item nobody remembers, and a CPU runtime does not
# announce itself — it trains at a rate that never finishes. On Kaggle
# the equivalent oversight cost an hour before anything said why, so this
# asserts rather than prints.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit("warn: no GPU on this runtime, so training would never finish. "
                     "fix: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("ok: GPU", gpu.stdout.strip())

In [ ]:
import glob, os, shutil
# Where the toolkit expects to find them, whichever source wins.
target = "/content/pub4/STUDIO/lora/ragnhild/dataset"
drive_set = "/content/drive/MyDrive/lora/ragnhild/dataset"

def count(path):
    return len(glob.glob(os.path.join(path, "*.jpg")) +
               glob.glob(os.path.join(path, "*.jpeg")) +
               glob.glob(os.path.join(path, "*.png")) +
               glob.glob(os.path.join(path, "*.webp")))

if count(drive_set):
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if os.path.islink(target) or os.path.isfile(target):
        os.remove(target)
    elif os.path.isdir(target):
        shutil.rmtree(target)
    shutil.copytree(drive_set, target)
    print("ok: dataset from Drive —", count(target), "images (not from the public repo)")
elif count(target):
    print("ok: dataset from the checkout —", count(target), "images")
    print("note: these images are public, because https://github.com/anon987654321/pub4.git is.")
else:
    raise SystemExit(
        "warn: no training images.\n"
        "fix: upload the captioned dataset to Drive at MyDrive/lora/ragnhild/dataset "
        "(images plus their .txt captions), then Run all again."
    )

# Every image needs its caption. ai-toolkit resolves <stem>.txt and falls
# through to the empty string when it is absent, so a broken pair is not an
# error there — it is an uncaptioned training image, and the run reports
# nothing.
stems = {os.path.splitext(os.path.basename(p))[0]
         for p in glob.glob(os.path.join(target, "*"))
         if not p.endswith(".txt")}
captions = {os.path.splitext(os.path.basename(p))[0]
            for p in glob.glob(os.path.join(target, "*.txt"))}
missing = sorted(stems - captions)
if missing:
    raise SystemExit("warn: %d image(s) have no .txt caption: %s\n"
                     "fix: upload the captions alongside the images."
                     % (len(missing), ", ".join(missing[:6])))
print("ok: every image is captioned")

In [ ]:
# Everything past here is Ruby. cuda_t4 is a render_config.rb profile:
# fp16 because Turing has no bf16, quantised because 16 GB will not hold
# FLUX.1-dev otherwise, 512 buckets because the budget is the clock.
os.environ["LORA_STEPS"] = "1000"
os.environ["LORA_SAMPLE_EVERY"] = "500"
subprocess.run(["ruby", "/content/pub4/STUDIO/lora/_toolkit/colab_session.rb", "ragnhild"],
               check=True)